In [41]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


# ============================================
# 1. Swiss roll 数据生成 + 高维嵌入
# ============================================

def generate_swiss_roll(n_samples, D=32, noise=0.0, device="cpu"):
    """
    生成 Swiss roll:
      - 参数 (u, v)
      - 3D: (x1, x2, x3)
      - 高维嵌入: A @ x_3d, A ∈ R^{D x 3} 正交列

    返回:
      X_high: (n_samples, D)
      y:      (n_samples,)  -- 一个光滑的 target（这里用 sin(u) 做 toy）
      u, v:   (n_samples,)  -- 方便你以后做 intrinsic 对比
    """
    u = torch.empty(n_samples, device=device).uniform_(3 * math.pi, 9 * math.pi)
    v = torch.empty(n_samples, device=device).uniform_(0.0, 20.0)

    x1 = u * torch.cos(u)
    x2 = v
    x3 = u * torch.sin(u)

    X3 = torch.stack([x1, x2, x3], dim=1)  # (n, 3)

    if noise > 0:
        X3 = X3 + noise * torch.randn_like(X3)

    # 随机高维正交嵌入
    # A: (D, 3), 列正交
    A = torch.randn(D, 3, device=device)
    # QR 分解，取 Q 的前 3 列
    Q, _ = torch.linalg.qr(A, mode="reduced")  # Q: (D, 3)
    X_high = X3 @ Q.T  # (n, 3) @ (3, D) -> (n, D)

    # toy 的回归目标：y = sin(u) / u 之类的平滑函数
    y = torch.sin(u) / (u + 1.0)

    return X_high, y, u, v


# ============================================
# 2. Hilbert 射影度量（正锥 R^m_{>0}）
# ============================================

def hilbert_distance(x, y, eps=1e-8):
    """
    x, y: (m,) 正向量（>0）
    d_H(x, y) = log( max_i x_i / y_i ) - log( min_i x_i / y_i )

    返回: float
    """
    # 避免 0
    x_safe = x.clamp_min(eps)
    y_safe = y.clamp_min(eps)
    
    ratio = x_safe / y_safe         # 不再二次 mask
    max_r = ratio.max()
    min_r = ratio.min()
    return (max_r.log() - min_r.log()).item()


# ============================================
# 3. 简单正锥模型：正权重线性回归
#    w = softplus(theta) ∈ R^D_{>0}
# ============================================

# class PositiveLinear(nn.Module):
#     """
#     y_hat = X @ w, 其中 w = softplus(theta) > 0
#     这样参数 w 始终在正锥内部，可以监控 Hilbert metric。
#     """

#     def __init__(self, D):
#         super().__init__()
#         # 可训练参数 theta ∈ R^D
#         self.theta = nn.Parameter(torch.zeros(D))

#     def forward(self, X):
#         # softplus 保证正性
#         w = F.softplus(self.theta)
#         return X @ w

#     def positive_params_vector(self):
#         # 把当前的 “正锥参数” 摊平成一个向量，用于算 Hilbert 距离
#         w = F.softplus(self.theta)
#         return w.detach().clone()

# ============================================
# 3. non-expansiive 正锥模型：非负权重线性回归
#   w = ReLU(theta) ∈ R^D_{≥0}
# For all 0 point, we recorded but vanished them in the hilbert distance calculation.
# ============================================
class ReLUPositiveLinear(nn.Module):
    def __init__(self, D):
        super().__init__()
        # 用小正数初始化，避免一开始就死在 ReLU 的 0 上
        theta0 = 0.1 * torch.ones(D)
        self.theta = nn.Parameter(theta0)

    def forward(self, X):
        w = F.relu(self.theta)
        return X @ w

    def relu_params_vector(self):
        return F.relu(self.theta).detach().clone()




In [42]:
def run_experiment_on_subset(
    X_full, y_full, n,
    num_epochs=500,
    lr=1e-2,
    l2_reg=1e-3,
    device="cpu"
):
    N, D = X_full.shape
    idx = torch.randperm(N, device=device)[:n]
    X = X_full[idx]
    y = y_full[idx]

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=n, shuffle=False)  # full-batch

    model = ReLUPositiveLinear(D).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    param_traj = []
    loss_traj = []

    for epoch in range(num_epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()

            # 1. 预测
            y_pred = model(batch_X)

            # 2. MSE 部分
            mse = F.mse_loss(y_pred, batch_y)

            # 3. L2 正则直接在 ReLU 后
            w = F.relu(model.theta)
            l2 = (w ** 2).sum()
            loss = mse + l2_reg * l2


            loss.backward()
            optimizer.step()

        loss_traj.append(loss.item())
        param_traj.append( model.relu_params_vector().cpu())

    w_star = param_traj[-1]
    w_init = param_traj[0]

    hilbert_to_final = []
    hilbert_to_init = []
    hilbert_between = []

    # prev = param_traj[0]
    # for w_t in param_traj:
    #     hilbert_to_final.append(hilbert_distance(w_t, w_star))
    #     hilbert_to_init.append(hilbert_distance(w_t, w_init))
    #     hilbert_between.append(hilbert_distance(w_t, prev))
    #     prev = w_t

    return {
        "n": n,
        "loss_traj": loss_traj,
        "hilbert_to_final": hilbert_to_final,
        "hilbert_to_init": hilbert_to_init,
        "hilbert_between": hilbert_between,
        "w_star": w_star,
        "param_traj": param_traj,
    }


In [43]:
def analyze_trajectory_with_relu(param_traj, eps=1e-8, threshold=1e-8):
    w_star = param_traj[-1]
    w_init = param_traj[0]

    mask = (w_star > threshold)
    if mask.sum() == 0:
        raise ValueError("w_star 的支撑为空，阈值 threshold 太大了。")

    def project(w):
        w_proj = w[mask]
        return w_proj.clamp_min(eps)

    w_star_proj = project(w_star)
    w_init_proj = project(w_init)

    hilbert_to_final = []
    hilbert_to_init = []
    hilbert_between = []

    prev_proj = project(param_traj[0])
    for t, w_t in enumerate(param_traj):
        w_proj = project(w_t)

        # 如果你想 debug 可以顺便检查
        if (w_proj <= eps).all():
            print(f"[warn] step {t}: all projected coords ≈ 0 (after clamp -> eps)")

        hilbert_to_final.append(hilbert_distance(w_proj, w_star_proj))
        hilbert_to_init.append(hilbert_distance(w_proj, w_init_proj))
        hilbert_between.append(hilbert_distance(w_proj, prev_proj))
        prev_proj = w_proj

    return hilbert_to_final, hilbert_to_init, hilbert_between


In [44]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    # ----- 生成大样本 Swiss roll -----
    N = 10_000
    D = 32
    X_full, y_full, u, v = generate_swiss_roll(
        n_samples=N,
        D=D,
        noise=0.0,
        device=device,
    )
    print(f"Full dataset: X={X_full.shape}, y={y_full.shape}")

    # ----- 不同小样本规模 -----
    n_list = [50, 100, 200, 500]
    num_epochs = 500
    lr = 1e-2
    l2_list = [0.0, 1e-4, 1e-3, 1e-2]

    # all_results[n][l2_reg] = 对应的实验结果
    all_results = {}

    for n in n_list:
        all_results[n] = {}
        print(f"\n================ n = {n} ================")

        for l2_reg in l2_list:
            print(f"\n--- Running experiment (n={n}, l2_reg={l2_reg}) ---")
            res = run_experiment_on_subset(
                X_full,
                y_full,
                n=n,
                num_epochs=num_epochs,
                lr=lr,
                l2_reg=l2_reg,
                device=device,
            )
            all_results[n][l2_reg] = res
            analysis = analyze_trajectory_with_relu(res["param_traj"])
            hilbert_to_final, hilbert_to_init, hilbert_between = analysis
            print(f"Final loss: {res['loss_traj'][-1]:.6f}")
            print(f"Initial d_H(w_t, w*): {hilbert_to_final[0]:.6f}")
            print(f"Final   d_H(w_t, w*): {hilbert_to_final[-1]:.6f}")
            print("First 15 d_H(w_t, w*):", hilbert_to_final[:15])
            print("First 15 d_H(w_t, w_0):", hilbert_to_init[:15])
            print("First 15 d_H(w_{t+1}, w_t):", hilbert_between[:15])

    # 例如保存结果
    torch.save(all_results, "swiss_roll_cone_mse_loss.pt")


In [45]:
main()

Using device: cuda
Full dataset: X=torch.Size([10000, 32]), y=torch.Size([10000])

================ n = 50 ================

--- Running experiment (n=50, l2_reg=0.0) ---
Final loss: 0.000547
Initial d_H(w_t, w*): 0.788763
Final   d_H(w_t, w*): 0.000000
First 15 d_H(w_t, w*): [0.788763165473938, 1.3319289684295654, 1.8534897565841675, 2.87717342376709, 2.5863447189331055, 2.567718744277954, 2.401895046234131, 2.2005038261413574, 2.005612850189209, 1.8284966945648193, 1.6701843738555908, 1.5288714170455933, 1.4022868871688843, 1.2883689403533936, 1.185401439666748]
First 15 d_H(w_t, w_0): [0.0, 0.572837233543396, 1.7527523040771484, 2.119800567626953, 2.216783046722412, 2.20658278465271, 1.9342795610427856, 1.6395474672317505, 1.3841943740844727, 1.1714847087860107, 0.9938426613807678, 0.8436170816421509, 0.7148269414901733, 0.6030497550964355, 0.505034863948822]
First 15 d_H(w_{t+1}, w_t): [0.0, 0.572837233543396, 1.8394980430603027, 3.557185173034668, 1.995356559753418, 0.265868693590